<a href="https://colab.research.google.com/github/eco-bios/monitoreo-deforestacion-chiapas/blob/main/Pipeline_Monitoreo_Poligonos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este Script permite autenticar la cuenta e inicializa la conexion con el proyecto y se verifica mediante un dataset la conexion a los servidores de google earth

In [ ]:
import ee
import warnings
# Esto ignora los mensajes de advertencia que no son críticos
warnings.filterwarnings('ignore')

# --- AUTENTICACIÓN ---
# Inicia el proceso de autorización para acceder a la cuenta de Earth Engine.
ee.Authenticate()

try:
    # --- INICIALIZACIÓN ---
    # Intenta establecer conexión con el proyecto específico en Google Cloud
    ee.Initialize(project='monitoreo-deforestacion-ch')

    # --- PRUEBA DE CONEXIÓN ---
    # Se carga un dataset global (SRTM) para verificar el acceso a los servidores
    test_image = ee.Image("USGS/SRTMGL1_003")
    print("✅ ¡CONECTADO! El motor de Google Earth Engine ya te reconoce.")

except Exception as e:
    # Mensaje en caso de fallo en las credenciales o el ID del proyecto
    print(f"❌ Error detallado: {e}")

Esta función automatiza la obtención de salud vegetal (NDVI) comparando dos periodos. Aplica filtros de nubosidad y promedia los valores en un radio de 1 km para asegurar resultados representativos a nivel de paisaje. Ademas puede utilizarse en distintos periodos y de forma escalable.

In [ ]:
def analizar_salud_vegetal(geometria, nombre_sitio):
    """
    Analiza la salud vegetal (NDVI) comparando 2024 vs 2026.
    Acepta puntos o polígonos de Earth Engine.
    """
    # 1. Ajuste de geometría usando el método nativo de GEE .type()
    # Si el tipo es 'Point', hacemos el buffer. Si no, usamos la geometría tal cual.
    tipo_geo = geometria.type().getInfo()
    region_analisis = geometria.buffer(1000) if tipo_geo == 'Point' else geometria

    # 2. Función interna para obtener NDVI
    def get_ndvi(fecha_inicio, fecha_fin):
        coleccion = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(region_analisis)
            .filterDate(fecha_inicio, fecha_fin)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

        mosaico = coleccion.median()
        return mosaico.normalizedDifference(['B8', 'B4']).rename('nd')

    # 3. Procesamiento y Estadísticas
    ndvi_24 = get_ndvi('2024-01-01', '2024-01-31')
    ndvi_26 = get_ndvi('2026-01-01', '2026-01-31')

    stats_24 = ndvi_24.reduceRegion(ee.Reducer.mean(), region_analisis, 10).get('nd').getInfo()
    stats_26 = ndvi_26.reduceRegion(ee.Reducer.mean(), region_analisis, 10).get('nd').getInfo()

    # 4. Reporte
    if stats_24 and stats_26:
        dif = stats_26 - stats_24
        print(f"📍 {nombre_sitio} | 2024: {stats_24:.3f} | 2026: {stats_26:.3f} | Delta: {dif:.3f}")
    else:
        print(f"❌ Datos insuficientes para {nombre_sitio}")

    return ndvi_26

En esta sección se invoca la función para diferentes ubicaciones de interés en Chiapas. Gracias a la modularidad del código, podemos monitorear ecosistemas variados (desde humedales hasta selva) simplemente actualizando las coordenadas decimales.

In [ ]:
# Para un PUNTO:
sitio_punto = ee.Geometry.Point([-92.60, 15.29])
analizar_salud_vegetal(sitio_punto, "Escuintla (Punto)")

# Para un POLÍGONO:
sitio_poligono = ee.Geometry.Polygon([[
    [-91.72, 17.72], [-91.71, 17.72], [-91.71, 17.73], [-91.72, 17.73], [-91.72, 17.72]
]])
analizar_salud_vegetal(sitio_poligono, "Parcela Catazajá (Polígono)")

Generamos un mapa dinámico para inspeccionar visualmente la salud vegetal. Utilizamos la librería geemap para superponer el ráster de NDVI sobre la ubicación analizada, permitiendo una validación visual de los datos calculados.

In [ ]:
# --- VISUALIZADOR DE POLÍGONO ---

# 1. Centramos el mapa en las coordenadas de la parcela
# Ajusta las coordenadas al centro de tu polígono
Map = geemap.Map(center=[17.725, -91.715], zoom=15)

# 2. Parámetros de visualización (los que ya definimos)
parametros_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#e74c3c', '#f1c40f', '#2ecc71']
}

# 3. Agregamos las capas
# ndvi_2026 es la variable que devolvió tu función
Map.addLayer(ndvi_2026, parametros_vis, 'Salud Vegetal (Enero 2026)')

# AQUÍ ESTÁ EL TRUCO: Agregamos el objeto de geometría directamente
# Usamos un estilo con color de borde azul y sin relleno (o relleno transparente)
Map.addLayer(sitio_poligono, {'color': '#3498db', 'fillColor': '00000000'}, 'Límites del Predio')

# 4. Mostrar el mapa
Map

Exportación de Resultados a Google Drive
Para asegurar la persistencia de los datos y permitir análisis posteriores en software SIG externo (como QGIS), exportamos el ráster de NDVI procesado. Se define una región rectangular que abarca el área de estudio y se utiliza el formato Cloud Optimized GeoTIFF para optimizar el rendimiento del archivo.

In [ ]:
# 1. Definición del área de exportación
# Generamos un área rectangular (bounding box) de 2km x 2km para el archivo de salida.
geometria_exportar = punto_escuintla.buffer(2000).bounds()

# 2. Configuración de la tarea de exportación a Google Drive
tarea = ee.batch.Export.image.toDrive(
    image=ndvi_2026,
    description='NDVI_SanJuanPanama_2026',
    folder='Proyecto_Monitoreo_Chiapas',
    scale=10,                      # Resolución espacial de Sentinel-2
    region=geometria_exportar,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True}
)

# 3. Ejecución y monitoreo de la tarea
# La tarea se procesa en los servidores de Google Earth Engine de forma asíncrona.
print("🚀 Iniciando exportación a Google Drive...")
tarea.start()

import time
while tarea.active():
    print(f"⌛ Estado de la tarea: {tarea.status()['state']}...", end="\r")
    time.sleep(15)

print(f"\n✅ Proceso completado. Archivo disponible en la carpeta 'Proyecto_Monitoreo_Chiapas'.")